In [4]:
import pandas as pd
import numpy as np

# Create a sample DataFrame 'df' with various data types and scenarios
data = {
    'CustomerID': [101, 102, 103, 101, 104, 105, 106, 107, 108, 109],
    'Email': ['a@example.com', 'b@example.com', 'c@example.com', 'a@example.com', 'd@example.com', 'e@example.com', 'f@example.com', 'g@example.com', 'h@example.com', 'i@example.com'],
    'OrderDate': ['01/15/2024', '2024-01-16', 'Feb 17 2024', '01/15/2024', '2024-02-18', '03/01/2024', '2024-03-02', 'Apr 03 2024', '2024-04-04', '05/05/2024'],
    'City': ['mumbai', 'Mumbai', 'MUMBAI', 'Delhi', 'pune', 'Chennai', 'Mumbai', 'Delhi', 'Pune', 'Chennai'],
    'Name': [' John   Doe ', 'Jane Smith', ' Alice  Brown', 'John   Doe', 'Robert Green', 'Emily White', 'Michael Black', 'Sarah Blue', 'David Gray', 'Olivia Red'],
    'Active': ['yes', 'true', '1', 'no', 'false', '0', 'yes', 'true', '1', 'no'],
    'Phone': ['123-456-7890', '(987) 654-3210', '111.222.3333', '123-456-7890', '444 555 6666', '777-888-9999', '555-123-4567', '999-888-7777', '666-333-1111', '222-444-6666'],
    'Revenue': ['1,234.56', '500.25', '2,100.00', '1,234.56', '750.80', '1500.00', '300.50', '900.00', '1800.00', '600.00'],
    'Growth': ['10%', '5.5%', '12%', '10%', '8%', '7.5%', '15%', '3%', '9%', '6.2%'],
    'Age': [30, 25, 35, np.nan, 40, 28, 32, 45, 22, 38],
    'Score': [85.5, 90.0, 78.2, 85.5, 92.1, 70.0, 88.0, 75.0, 95.0, 81.0]
}
df = pd.DataFrame(data)

print("Sample DataFrame 'df' created successfully:")
display(df.head())

Sample DataFrame 'df' created successfully:


,CustomerID,Email,OrderDate,City,Name,Active,Phone,Revenue,Growth,Age,Score
0,101,a@example.com,01/15/2024,mumbai,John Doe,yes,123-456-7890,"1,234.56",10%,30.0,85.5
1,102,b@example.com,2024-01-16,Mumbai,Jane Smith,true,(987) 654-3210,500.25,5.5%,25.0,90.0
2,103,c@example.com,Feb 17 2024,MUMBAI,Alice Brown,1,111.222.3333,"2,100.00",12%,35.0,78.2
3,101,a@example.com,01/15/2024,Delhi,John Doe,no,123-456-7890,"1,234.56",10%,NaN,85.5
4,104,d@example.com,2024-02-18,pune,Robert Green,false,444 555 6666,750.80,8%,40.0,92.1


In [5]:
import pandas as pd

# ── Detect duplicates ──────────────────────────────────────────────
print('Total full duplicates:', df.duplicated().sum())
print('Duplicates on key columns:',
       df.duplicated(subset=['CustomerID', 'Email']).sum())

# View duplicate rows (both original and duplicate shown)
print(df[df.duplicated(keep=False)].sort_values('CustomerID'))

# ── Remove duplicates ──────────────────────────────────────────────
# Keep first occurrence
df_clean = df.drop_duplicates(keep='first')

# Keep last occurrence (e.g., most recent update)
df_clean = df.drop_duplicates(subset=['CustomerID'], keep='last')

print(f'Before: {len(df)} rows | After: {len(df_clean)} rows')
print(f'Removed: {len(df) - len(df_clean)} duplicate records')


Total full duplicates: 0
Duplicates on key columns: 1
Empty DataFrame
Columns: [CustomerID, Email, OrderDate, City, Name, Active, Phone, Revenue, Growth, Age, Score]
Index: []
Before: 10 rows | After: 9 rows
Removed: 1 duplicate records


In [7]:
import pandas as pd

# Mixed formats: '01/15/2024', '2024-01-15', 'Jan 15 2024'
df['OrderDate'] = pd.to_datetime(df['OrderDate'], format='mixed', dayfirst=False)

# Extract useful components
df['Year']    = df['OrderDate'].dt.year
df['Month']   = df['OrderDate'].dt.month
df['Quarter'] = df['OrderDate'].dt.quarter
df['DayName'] = df['OrderDate'].dt.day_name()

print(df[['OrderDate','Year','Month','Quarter']].head())

   OrderDate  Year  Month  Quarter
0 2024-01-15  2024      1        1
1 2024-01-16  2024      1        1
2 2024-02-17  2024      2        1
3 2024-01-15  2024      1        1
4 2024-02-18  2024      2        1


In [8]:
# Fix case inconsistency: 'mumbai', 'Mumbai', 'MUMBAI' -> 'Mumbai'
df['City'] = df['City'].str.strip().str.title()

# Fix whitespace and special characters
df['Name'] = df['Name'].str.strip().str.replace(r'\s+', ' ', regex=True)

# Standardise boolean-like columns
df['Active'] = df['Active'].str.lower().map({
    'yes': 1, 'no': 0, 'true': 1, 'false': 0, '1': 1, '0': 0
})

# Standardise phone numbers
df['Phone'] = df['Phone'].str.replace(r'[^\d]', '', regex=True)
df['Phone'] = df['Phone'].apply(lambda x: x[-10:] if len(x)>=10 else x)


In [9]:
# Convert string numbers to numeric
df['Revenue'] = pd.to_numeric(df['Revenue'].str.replace(',',''),
                              errors='coerce')

# Convert percentage strings to float
df['Growth'] = df['Growth'].str.rstrip('%').astype(float) / 100

# Downcast to reduce memory
df['Age']    = df['Age'].astype('Int32')   # Nullable integer
df['Score']  = df['Score'].astype('float32')


In [10]:
from sklearn.preprocessing import LabelEncoder

data = ['Mumbai', 'Pune', 'Delhi', 'Mumbai', 'Chennai', 'Pune']
le = LabelEncoder()
encoded = le.fit_transform(data)
print(encoded)        # Output: [2 3 1 2 0 3]
print(le.classes_)    # Output: ['Chennai' 'Delhi' 'Mumbai' 'Pune']

# Reverse transform
print(le.inverse_transform([2, 0]))  # Output: ['Mumbai' 'Chennai']


[2 3 1 2 0 3]
['Chennai' 'Delhi' 'Mumbai' 'Pune']
['Mumbai' 'Chennai']


In [11]:
import pandas as pd

df = pd.DataFrame({'City': ['Mumbai','Pune','Delhi','Mumbai']})

# Using pandas get_dummies
df_encoded = pd.get_dummies(df, columns=['City'], prefix='City')
print(df_encoded)

# Output:
#    City_Delhi  City_Mumbai  City_Pune
# 0           0            1          0
# 1           0            0          1
# 2           1            0          0
# 3           0            1          0

# Drop first to avoid dummy variable trap
df_encoded = pd.get_dummies(df, columns=['City'], drop_first=True)


   City_Delhi  City_Mumbai  City_Pune
0       False         True      False
1       False        False       True
2        True        False      False
3       False         True      False
